# **UNIWAY: AR NAVIGATION APPLICATION FOR UMM AL-QURA UNIVERSITY**
### **Centralized Cloud Backend API Architecture (FastAPI & Cloud Firestore)**

This notebook hosts the production-ready centralized backend API infrastructure for the **UniWay** application (Al-Zahir Campus). The system is engineered as a high-performance, asynchronous service layer that unifies computer vision intelligence with secure, isolated cloud database repositories across four core functional layers:

1. **Machine Learning Inference Engine**: Executes spatial object localization via custom-trained **YOLOv8** networks and bilingual character parsing via **EasyOCR**, augmented by an advanced LAB-space CLAHE image enhancement pipeline.
2. **Dynamic Search & Navigation Semantics**: Manages continuous, high-tolerance text lookups with Arabic query normalization to evaluate and match inputs against Firestore keys and fields simultaneously.
3. **Anonymous Data Privacy & Isolation**: Enforces strict student privacy by mapping custom classroom bookmark arrays dynamically via unique hardware device identifiers inside **Cloud Firestore**, avoiding personal data collection.
4. **Live Gateway Synchronization**: Establishes secure, public HTTP tunneling via **Ngrok** to expose live ports for interactive mobile frontend testing and validation.

## **0. Environmental Initialization & Dynamic Resource Provisioning**
This foundational section deploys the operational dependency framework, importing requisite core deep learning libraries, asynchronous routing gateways, and establishing secure persistent database validation structures.

* **Core Functions:**
  * **Dependency Deployment**: Provisions runtime package requirements including execution loops, server gateways, computer vision frameworks, and linguistic character parsers quietly.
  * **Defensive Environment Auditing**: Executes an automated systemic check to evaluate the local host environment. It dynamically isolates Google Colab runtime mount execution tracks to shield local or cloud-native instances (e.g., Render) from invalid module initialization exceptions.

In [ ]:
# 1. Install all required packages quietly
!pip install fastapi uvicorn ultralytics python-multipart nest-asyncio pyngrok easyocr firebase-admin opencv-python-headless -q
!pip install ipynb -q

# 2. Import core system dependencies natively
import os
import sys
import nest_asyncio
import cv2
import numpy as np
from fastapi import FastAPI, Header, HTTPException, status, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# 3. Import Firebase cloud administrative toolkits
import firebase_admin
from firebase_admin import credentials, firestore

# 4. Conditional defensive authorization for Google Colab environments
is_colab = 'google.colab' in sys.modules or 'IPython' in sys.getcounter().keys() if hasattr(sys, 'getcounter') else os.path.exists('/content')

if is_colab:
    try:
        from google.colab import drive
        # Target path validation to prevent redundant multi-mount collision locks
        if not os.path.exists('/content/drive/MyDrive'):
            print("[STATUS] Initiating secure Google Drive storage validation tunnel...")
            drive.mount('/content/drive')
        else:
            print("[STATUS] Google Drive filesystem is already securely mounted and accessible.")
    except Exception as drive_err:
        print(f"[WARNING] Google Drive automated mounting failed: {str(drive_err)}")
        print("[STATUS] Process continuing. Ensure manual directory structures are provisioned.")
else:
    print("[STATUS] Non-Colab localized environment detected. Skipping Google Drive structural binding.")

print("\nEnvironment Ready: Core AI dependencies deployed, cloud libraries instantiated.")

[STATUS] Google Drive filesystem is already securely mounted and accessible.

Environment Ready: Core AI dependencies deployed, cloud libraries instantiated.


## **1. Machine Learning Inference & Visual Recognition System**
This section implements the core artificial intelligence computer vision pipeline. It manages image file binary streams transmitted from the mobile camera framework to perform spatial object detection and bilingual optical character recognition (OCR).

* **Endpoints:**
  * **GET** `/`: Health check gateway that returns the active server status, project identity, and structural campus location context.
  * **POST** `/predict`: Accepts live camera image frames, executes localized object detection via YOLOv8, enhances visual data using CLAHE, runs EasyOCR parsing, and queries the matched Firestore target.

* **Core Functions & Preprocessing Pipeline:**
  * **Signage Localization**: Utilizes a custom-trained **YOLOv8** deep learning model (`best_uqu_v1.pt`) to detect signboards, filtering out background noise with a strict confidence threshold guardrail.
  * **LCDM Enhancement (Local Contrast & Detail Management)**: Transforms cropped images via a custom-coded preprocessing pipeline that executes Contrast Limited Adaptive Histogram Equalization (**CLAHE**) across the **LAB color space** to neutralize bad lighting and blur.
  * **Bilingual Structural Character Parsing**: Deploys an **EasyOCR** engine to run linguistic recognition in both Arabic and English (`['ar', 'en']`) directly over GPU-accelerated memory layers.
  * **Dynamic Structural Formatting**: Re-maps raw text strings, strips whitespace, converts Eastern digits to standard formats, isolates alphanumeric structures, and handles missing localized properties before performing direct lookups inside the **Cloud Firestore `ClassRoom` collection**.

In [ ]:
# ===========================================================================
# 1: SYSTEM INITIALIZATION, CLOUD CONNECTIONS & RECOGNITION MODELS
# ===========================================================================
import torch
from ultralytics import YOLO
import easyocr
import cv2
import numpy as np
from PIL import Image
import nest_asyncio
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
import io
import re
import os
import firebase_admin
from firebase_admin import credentials, firestore

# Enable nested event loops for Google Colab environment compatibility
nest_asyncio.apply()

# Globally initialize core dependency holders as None to prevent runtime NameErrors
db = None
detection_model = None
reader = None

# 1. Initialize Google Cloud Firebase Admin SDK with robust pathway protection
FIREBASE_KEY_PATH = '/content/drive/MyDrive/UniWay_Final_System/Firebase/serviceAccountKey.json'
try:
    if not firebase_admin._apps:
        if not os.path.exists(FIREBASE_KEY_PATH):
            raise FileNotFoundError(f"Firebase credential file missing at expected path: {FIREBASE_KEY_PATH}")
        cred = credentials.Certificate(FIREBASE_KEY_PATH)
        firebase_admin.initialize_app(cred)
    db = firestore.client()
    print("[STATUS] Google Cloud Firestore communication bridge successfully active.")
except Exception as e:
    print(f"[CRITICAL] Firebase distributed credential sync failure: {str(e)}")

# 2. Load Deep Learning Weights for Signage Object Localization (YOLOv8) safely
DETECTION_PATH = '/content/drive/MyDrive/UniWay_Final_System/Weights/best_uqu_v1.pt'
try:
    if not os.path.exists(DETECTION_PATH):
        raise FileNotFoundError(f"YOLOv8 weight file missing at expected path: {DETECTION_PATH}")
    detection_model = YOLO(DETECTION_PATH)
    print("[STATUS] Custom Trained YOLOv8 Spatial Detection Network loaded.")
except Exception as e:
    print(f"[CRITICAL] YOLOv8 deep learning initialization failure: {str(e)}")

# 3. Load Multilingual Linguistic Recognition Engine (EasyOCR) safely
try:
    reader = easyocr.Reader(['ar', 'en'], gpu=torch.cuda.is_available())
    print("[STATUS] Bilingual Structural Character Parser successfully mounted in GPU/CPU memory.")
except Exception as e:
    print(f"[CRITICAL] EasyOCR linguistic framework initialization failure: {str(e)}")

# 4. Initialize Core Web Gateway (FastAPI Framework Configuration)
app = FastAPI(title="UniWay Navigation Core Routing API", version="1.0.0")

# Configure Cross-Origin Resource Sharing (CORS) parameters for mobile clients
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
print("[STATUS] FastAPI routing layers successfully instantiated and configured.")

[STATUS] Google Cloud Firestore communication bridge successfully active.
[STATUS] Custom Trained YOLOv8 Spatial Detection Network loaded.
[STATUS] Bilingual Structural Character Parser successfully mounted in GPU/CPU memory.
[STATUS] FastAPI routing layers successfully instantiated and configured.


In [ ]:
# ===========================================================================
# 2: FIRESTORE-COMPATIBLE NORMALIZATION PIPELINE
# ===========================================================================
import os
import sys
import pandas as pd
import re
import cv2
import numpy as np
from PIL import Image

# Initialize the ground truth dataframe as an empty DataFrame to guarantee safety
ground_truth_df = pd.DataFrame(columns=['RoomID'])

TARGET_SCRIPT_DIR = '/content/drive/MyDrive/UniWay_Final_System/Notebooks'
SCRIPT_FILE_PATH = os.path.join(TARGET_SCRIPT_DIR, 'UniWay_Recognition_OCR.py')

def your_custom_preprocessing_pipeline(cropped_image_pil):
    """
    Applies your exact verified LCDM Enhancement (Local Contrast & Detail Management).
    Enhances the image using CLAHE in LAB color space based on your custom dataset code.
    Wrapped inside safety layers to intercept empty or corrupted image arrays.
    """
    try:
        if cropped_image_pil is None:
            return None

        # Convert PIL Image safely to OpenCV format (BGR)
        opencv_img = np.array(cropped_image_pil)
        if opencv_img.size == 0:
            return None

        opencv_img = cv2.cvtColor(opencv_img, cv2.COLOR_RGB2BGR)

        # Convert to LAB color space to isolate the Lightness channel
        lab = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)

        # Apply CLAHE safely
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        cl = clahe.apply(l)

        # Merge channels back and return as BGR image
        limg = cv2.merge((cl, a, b))
        enhanced_bgr = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
        return enhanced_bgr
    except Exception as e:
        print(f"[ERROR] LCDM Enhancement pipeline failure: {str(e)}")
        return None

def normalize_text(text):
    """
    Standardizes text to match UQU Firestore format (e.g., 'أ202', 'د104').
    Keeps Arabic letters to align with database keys but converts Hindi numbers to standard English digits.
    """
    if not text:
        return ""
    text = str(text).strip()

    # Mapping dictionary designed for UQU Firestore structural consistency
    replacements = {
        'ا': 'أ', 'إ': 'أ', 'آ': 'أ',
        '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5',
        '٦': '6', '٧': '7', '٨': '8', '٩': '9', '٠': '0',
        'A': 'أ', 'D': 'د'
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    cleaned_text = re.sub(r'[^\w\s]', '', text)
    cleaned_text = cleaned_text.replace(" ", "")
    return cleaned_text

def intelligent_fix(text):
    """
    Your exact Heuristic Intelligence from Phase 7.2 of UniWay_Recognition_OCR.py
    Standardizes prediction length based on UQU architectural patterns.
    """
    text = str(text).strip()
    if any(char.isdigit() for char in text) and len(text) > 4:
        return text[0] + text[-3:]
    return text

def run_uniway_engine_api(img_path, database_df, current_reader):
    """
    Your exact 77.27% verified Inference Engine from Phase 3 of your code.
    Adapted safely to route through FastAPI context with defensive structure fallbacks.
    """
    try:
        if not os.path.exists(img_path) or database_df is None or database_df.empty:
            return {
                'Filename': os.path.basename(img_path) if img_path else "Unknown",
                'Raw_OCR': "", 'Building_Context': "", 'Status': 'FAILED', 'Prediction': 'Unknown'
            }

        filename = os.path.basename(img_path)
        img = cv2.imread(img_path)
        if img is None:
            return {'Filename': filename, 'Raw_OCR': "", 'Building_Context': "", 'Status': 'FAILED', 'Prediction': 'Unknown'}

        # Apply your official LCDM Enhancement safely from PIL image
        pil_img = Image.open(img_path)
        enhanced = your_custom_preprocessing_pipeline(pil_img)
        if enhanced is None:
            enhanced = img # Fallback to raw BGR if enhancement fails

        # Extract Building Identifier from Spatial Context
        target_building = ""
        if '_A' in filename.upper(): target_building = 'أ'
        elif '_D' in filename.upper(): target_building = 'د'

        # OCR Recognition using bilingual reader
        results = current_reader.readtext(enhanced, detail=1) if current_reader else []
        raw_text = " ".join([res[1] for res in results]) if results else ""

        # Standardized Normalization
        normalized_detected = normalize_text(raw_text)

        # Database Matching Loop with validation guards
        match = database_df[
            (database_df['RoomID'].apply(lambda x: normalize_text(str(x)) == normalized_detected if pd.notna(x) and normalized_detected != "" else False)) |
            (database_df['RoomID'].apply(lambda x: normalize_text(str(x)) == intelligent_fix(normalized_detected) if pd.notna(x) and normalized_detected != "" else False))
        ]

        log = {
            'Filename': filename,
            'Raw_OCR': raw_text,
            'Building_Context': target_building,
            'Status': 'FAILED',
            'Prediction': 'Unknown'
        }

        if not match.empty:
            info = match.iloc[0]
            log['Status'] = 'SUCCESS'
            log['Prediction'] = str(info['RoomID'])

        return log
    except Exception as e:
        print(f"[ERROR] Engine inference pipeline breakdown: {str(e)}")
        return {'Filename': "InferenceError", 'Raw_OCR': "", 'Building_Context': "", 'Status': 'FAILED', 'Prediction': 'Unknown'}

# Load your official Excel database from its final directory defensively
DB_PATH = '/content/drive/MyDrive/UniWay_Final_System/Firebase/UQU_Class_DB.xlsx'
if not os.path.exists(DB_PATH):
    DB_PATH = '/content/drive/MyDrive/UniWay_Dataset/UQU_Class_DB.xlsx'

try:
    if os.path.exists(DB_PATH):
        ground_truth_df = pd.read_excel(DB_PATH)
        print(f"[STATUS] Official UQU Excel Database successfully synced from: {DB_PATH}")
    else:
        print(f"[WARNING] Excel path not found. Running with empty fallback context.")
except Exception as e:
    print(f"[WARNING] Excel Database compilation skipped/failed: {str(e)}")

print("[STATUS] Cell 2 Compiled Successfully with Firestore naming alignments.")

[STATUS] Official UQU Excel Database successfully synced from: /content/drive/MyDrive/UniWay_Dataset/UQU_Class_DB.xlsx
[STATUS] Cell 2 Compiled Successfully with Firestore naming alignments.


In [ ]:
# ===========================================================================
# 3: FASTAPI INFERENCE ENDPOINT WITH ADVANCED ARABIC TEXT CLEANING
# ===========================================================================
import io
import os
from fastapi import HTTPException, status, File, UploadFile

def normalize_ml_text(text: str) -> str:
    """
    Advanced Arabic text stripping to eliminate formatting traps (Hamzas, Ta-Marbuta, Spaces).
    Guarantees successful string intersections even with heavy OCR noise.
    """
    if not text:
        return ""

    # Convert to string, strip boundaries, and lowercase
    text = str(text).strip().lower()

    # Standardize all variations of Alef to a bare Alef 'ا'
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا").replace("آ", "ا")

    # Standardize Ta-Marbuta 'ة' to Heh 'ه' and Yeh variants
    text = text.replace("ة", "ه").replace("ى", "ي")

    # Remove all whitespace characters to handle concatenated text chunks
    text = text.replace(" ", "").replace("\t", "").replace("\n", "")
    return text

def enforce_uqu_room_constraints(digits: str) -> str:
    """
    Heuristic constraint enforcer specialized for UQU architectural room configurations.
    Intercepts OCR layout noise that appends or prepends faulty digits (e.g., converting 1051 to 105).
    """
    if not digits:
        return ""

    if len(digits) == 4:
        if digits.endswith('1'):
            return digits[:3]
        elif digits.startswith('1'):
            return digits[1:]
        return digits[:3]

    return digits

@app.get("/")
def root():
    return {
        "status": "Online",
        "project": "UniWay Indoor Navigation System",
        "campus": "Al-Zahir Campus - Umm Al-Qura University"
    }

@app.post("/predict")
async def predict_signage(file: UploadFile = File(...)):
    """
    Takes the uploaded mobile frame, crops the signage via YOLO,
    normalizes the OCR text, enforces strict digit constraints, and executes lookup loops.
    """
    try:
        # 1. Secure payload retrieval and execute dynamic format validation
        contents = await file.read()
        try:
            image = Image.open(io.BytesIO(contents)).convert("RGB")
        except Exception as img_err:
            return {
                "status": "error",
                "error_code": "INVALID_IMAGE_PAYLOAD",
                "message": f"Submitted binary file is corrupted or not a valid image: {str(img_err)}"
            }

        # 2. Check initialized framework constraints
        if detection_model is None:
            return {
                "status": "error",
                "error_code": "MODEL_UNAVAILABLE",
                "message": "YOLOv8 spatial detection layers are not currently loaded on the server."
            }

        # 3. Signage Localization via YOLOv8
        detection_results = detection_model(image, conf=0.3)
        if len(detection_results[0].boxes) == 0:
            return {
                "status": "error",
                "error_code": "NO_SIGNAGE_FOUND",
                "message": "Signage localization failed."
            }

        # 4. Crop the detected Signage with dimension security checks
        box = detection_results[0].boxes[0].xyxy[0].cpu().numpy().astype(int)
        if (box[2] <= box[0]) or (box[3] <= box[1]):
            return {
                "status": "error",
                "error_code": "INVALID_BOUNDING_BOX",
                "message": "Localization layout produced zero-width spatial dimensions."
            }

        cropped_img = image.crop((box[0], box[1], box[2], box[3]))

        # Save the crop temporarily using agnostic operating system pathway architectures
        temp_crop_path = os.path.join(os.getcwd(), "temp_crop_api.jpg")
        cropped_img.save(temp_crop_path)

        # 5. RUN PREPROCESSING: Apply your verified LCDM Enhancement safely
        enhanced_numpy = your_custom_preprocessing_pipeline(cropped_img)
        if enhanced_numpy is None:
            enhanced_numpy = np.array(cropped_img) # Structural fallback

        # 6. OCR RECOGNITION
        raw_text = ""
        if reader is not None:
            ocr_results = reader.readtext(enhanced_numpy, detail=1)
            raw_text = " ".join([res[1] for res in ocr_results]) if ocr_results else ""

            if not raw_text:
                raw_numpy = np.array(cropped_img)
                ocr_results_raw = reader.readtext(raw_numpy)
                raw_text = " ".join([res[1] for res in ocr_results_raw]) if ocr_results_raw else ""

        if not raw_text:
            return {
                "status": "error",
                "error_code": "OCR_FAILED",
                "message": "Signage detected, but text extraction failed."
            }

        # 7. RE-FORMAT TEXT TO MATCH YOUR FIRESTORE ID (Format: '105أ')
        hindi_to_eng = {'١':'1','٢':'2','٣':'3','٤':'4','٥':'5','٦':'6','٧':'7','٨':'8','٩':'9','٠':'0'}
        clean_str = str(raw_text).strip()
        for h, e in hindi_to_eng.items():
            clean_str = clean_str.replace(h, e)

        clean_str = clean_str.replace('ا', 'أ').replace('إ', 'أ').replace('آ', 'أ').replace('A', 'أ').replace('a', 'أ')
        clean_str = clean_str.replace('D', 'د').replace('d', 'د')

        # Isolate segments
        raw_digits = "".join(filter(str.isdigit, clean_str))
        letters_part = "".join(filter(lambda x: not x.isdigit(), clean_str)).replace(" ", "")

        # ENFORCE ARCHITECTURAL DIGIT RULES: Intercepts and corrects 4-digit layout malfunctions (e.g. 1051 -> 105)
        digits_part = enforce_uqu_room_constraints(raw_digits)

        if not letters_part:
            letters_part = "أ"

        target_room_id = f"{digits_part}{letters_part}"

        # 8. DIRECT LOOKUP IN CLOUD FIRESTORE WITH HIGH-TOLERANCE FUZZY MATCHING
        if db is None:
            return {
                "status": "error",
                "error_code": "FIRESTORE_OFFLINE",
                "message": "Active connection bridge to Google Cloud Firebase is unavailable."
            }

        try:
            # First Layer: Direct Document ID Lookup (Strict numeric check)
            doc_ref = db.collection('ClassRoom').document(target_room_id)
            doc = doc_ref.get()

            if doc.exists:
                data = doc.to_dict()
                return {
                    "status": "success",
                    "message": "Match Found directly via Document ID Lookup",
                    "data": {
                        "detected_text_raw": raw_text,
                        "processed_room_id": doc.id,
                        "className": doc.id,
                        "buildingId": data.get("buildingId", "Not Specified"),
                        "floorNum": data.get("floorNum", "Not Specified"),
                        "description": data.get("description", "No description available.")
                    }
                }

            # Second Layer: High-Tolerance Substring Matching (Normalized for text name fallbacks)
            normalized_ocr_query = normalize_ml_text(raw_text)
            all_classrooms = db.collection('ClassRoom').stream()

            for room_doc in all_classrooms:
                room_data = room_doc.to_dict()
                db_classname = room_data.get("classname", room_data.get("className", ""))
                normalized_db_classname = normalize_ml_text(db_classname)

                # HIGH-TOLERANCE CROSS-MATCHING: Triggers if either string encapsulates the other
                if normalized_db_classname and ((normalized_db_classname in normalized_ocr_query) or (normalized_ocr_query in normalized_db_classname)):
                    return {
                        "status": "success",
                        "message": "Match Found via High-Tolerance Classroom Name Attribute Lookup",
                        "data": {
                            "detected_text_raw": raw_text,
                            "processed_room_id": room_doc.id,
                            "className": db_classname,
                            "buildingId": room_data.get("buildingId", "Not Specified"),
                            "floorNum": room_data.get("floorNum", "Not Specified"),
                            "description": room_data.get("description", "No description available.")
                        }
                    }

            # Third Layer: Try with fallback space padding for ID
            fallback_id = f"{target_room_id} "
            doc_ref_fb = db.collection('ClassRoom').document(fallback_id)
            doc_fb = doc_ref_fb.get()

            if doc_fb.exists:
                data_fb = doc_fb.to_dict()
                return {
                    "status": "success",
                    "message": "Match Found directly via Document ID Lookup (with layout padding)",
                    "data": {
                        "detected_text_raw": raw_text,
                        "processed_room_id": doc_fb.id,
                        "className": target_room_id,
                        "buildingId": data_fb.get("buildingId", "Not Specified"),
                        "floorNum": data_fb.get("floorNum", "Not Specified"),
                        "description": data_fb.get("description", "No description available.")
                    }
                }

            # Final Fallback if no ID or Name match is found anywhere
            return {
                "status": "success",
                "message": "Generated search target but location could not be identified in current database schemas.",
                "data": {
                    "detected_text_raw": raw_text,
                    "processed_room_id": target_room_id,
                    "className": target_room_id,
                    "buildingId": "Unknown",
                    "floorNum": "Unknown",
                    "description": "Location key not found in current database mapping."
                }
            }

        except Exception as db_err:
            return {"status": "error", "error_code": "DATABASE_ERROR", "message": f"Firestore Fetch Error: {str(db_err)}"}

    except Exception as e:
        return {"status": "error", "error_code": "SERVER_ERROR", "message": str(e)}

## **2. Navigation & Flexible Search System**
This section manages classroom lookups directly from the Cloud Firestore database, handling queries from both the home screen and post-camera navigation flows.

* **Endpoints:**
  * **GET** `/classrooms/search`: A unified, high-tolerance search endpoint. It applies advanced Arabic text normalization and digit conversion to evaluate and match user queries against both Firestore Document IDs and internal fields simultaneously.

* **Core Functions:**
  * **Text Normalization**: Strips continuous spacing, standardizes all Alif/Ya variations, and converts Eastern Arabic numerals to Western digits to eliminate user typing friction.
  * **Dual-Layer Matching**: Dynamically scans document keys and field attributes to return partial (substring) results safely, preventing empty states and managing missing localized data.

In [ ]:
# --- Navigation & Search System ---

def normalize_arabic_text(text: str) -> str:
    """
    Cleans and normalizes text input to ensure flexible search matching.
    It strips surrounding whitespace, converts text to lowercase, standardizes
    Arabic characters (Alif, Ya, Ta Marbuta), and converts Eastern digits.
    """
    if not text:
        return ""

    # Remove leading and trailing spaces, and convert English characters to lowercase
    text = text.strip().lower()

    # Convert Eastern Arabic numerals (٢١٠) to standard Western digits (210)
    eastern_digits = "٠١٢٣٤٥٦٧٨٩"
    western_digits = "0123456789"
    digit_map = str.maketrans(eastern_digits, western_digits)
    text = text.translate(digit_map)

    # Standardize all forms of Alif
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")

    # Standardize Ya and Ta Marbuta
    text = text.replace("ى", "ي").replace("ة", "ه")

    # Remove any internal spaces to allow continuous matching
    text = text.replace(" ", "")

    return text


# Unified flexible endpoint handling text lookup for both search bars
@app.get("/classrooms/search")
async def search_classroom(className: str):
    try:
        classroom_collection = "ClassRoom".strip()
        normalized_query = normalize_arabic_text(className)

        # Fetch all classroom documents for flexible in-memory sub-string matching
        all_docs = db.collection(classroom_collection).stream()
        results = []

        for doc in all_docs:
            doc_data = doc.to_dict()

            # Extract document ID (e.g., "101" or "210") and normalize it
            doc_id = doc.id
            normalized_doc_id = normalize_arabic_text(doc_id)

            # Extract internal classname field and normalize it
            db_class_name = doc_data.get("classname", doc_data.get("className", ""))
            normalized_db_name = normalize_arabic_text(db_class_name)

            # Match against both the Document ID and the internal field for maximum safety
            if (normalized_query in normalized_doc_id) or (normalized_query in normalized_db_name):
                # Ensure document ID is included in the returned data for the frontend team
                doc_data["id"] = doc.id
                results.append(doc_data)

        return {
            "query": className,
            "results": results
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Firestore Error: {str(e)}")

## **3. Bookmark Configuration & Anonymous Data Isolation**
This section handles user bookmark management and enforces strict data isolation using unique hardware device identifiers to guarantee student privacy without collecting personal records.

* **Endpoints:**
  * **POST** `/bookmarks/add`: Receives a classroom identifier and custom description via client headers, securely creating or updating isolated user profiles and mapping the record to the target device.
  * **GET** `/bookmarks/my`: Queries the global storage schema to retrieve and return the complete array of saved classrooms exclusive to the requesting hardware client.

* **Core Functions:**
  * **Data Isolation**: Dynamically generates unique composite document IDs using a structured `{x_device_id}_{classId}` layout to eliminate multi-user data overwrites or cross-device leakage.
  * **Automated Provisioning**: Automatically intercepts baseline repository calls to check device persistence, dynamically provisioning active metadata state properties for new devices within the `User` collection.

In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore
from fastapi import FastAPI, Header, HTTPException, status

# Initialize Firebase Admin SDK connection to Firestore
if not firebase_admin._apps:
    cred = credentials.Certificate('/content/drive/MyDrive/UniWay/serviceAccountKey.json')
    firebase_admin.initialize_app(cred)

db = firestore.client()

# --- User Privacy & Data Isolation (Bookmark System) ---

# Endpoint to save a new classroom bookmark directly to the existing Bookmark collection
@app.post("/bookmarks/add")
async def add_bookmark(
    classId: str,
    description: str, # New field to accept room description from the app
    x_device_id: str = Header(None)
):
    if not x_device_id:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Device ID missing in headers. Data isolation cannot be enforced."
        )

    try:
        # Check and ensure the user record exists in the User collection
        user_collection = "User".strip()
        user_ref = db.collection(user_collection).document(x_device_id)
        if not user_ref.get().exists:
            user_ref.set({
                "isActive": True
            })

        # Sanitize the Bookmark collection name to prevent layout duplication
        bookmark_collection = "Bookmark".strip()

        # Construct a direct document identifier combining device ID and classroom ID to prevent conflicts
        doc_id = f"{x_device_id}_{classId}"

        # Save explicit fields directly into the existing Bookmark collection
        bookmark_ref = db.collection(bookmark_collection).document(doc_id)
        bookmark_ref.set({
            "classId": classId,
            "description": description, # Store description here to appear directly in Firestore
            "userId": x_device_id
        })

        return {
            "status": "success",
            "message": f"Successfully isolated and bookmarked Class ID: {classId} with description in your Bookmark table."
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Firestore Error: {str(e)}")


# Endpoint to fetch isolated bookmarks for the requesting device only
@app.get("/bookmarks/my")
async def get_my_bookmarks(x_device_id: str = Header(None)):
    if not x_device_id:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Device ID missing in headers."
        )

    try:
        # Sanitize collection name to prevent trailing spaces
        bookmark_collection = "Bookmark".strip()

        # Query the collection using a where clause to filter data by current userId
        bookmarks_ref = db.collection(bookmark_collection).where('userId', '==', x_device_id)
        docs = bookmarks_ref.stream()

        my_bookmarks = []
        for doc in docs:
            my_bookmarks.append(doc.to_dict())

        if not my_bookmarks:
            return {
                "userId": x_device_id,
                "bookmarks": [],
                "message": "No bookmarks saved for this user yet."
            }

        return {
            "userId": x_device_id,
            "bookmarks": my_bookmarks # List will automatically contain both classId and description
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Firestore Error: {str(e)}")

## **4. Live Gateway & Execution Tunnel**
This section initializes the asynchronous ASGI server loop and exposes the local application architecture to a secure, public environment to facilitate mobile frontend team evaluation and connectivity testing.

* **Core Endpoints Documented:**
  * **GET** `/docs`: Exposes the dynamic Swagger UI documentation containing interactive exploration tools for all backend routing layers.
  * **GET** `/classrooms/search`: Live route for unified character matching and continuous text search processes.

* **Core Functions:**
  * **Secure Tunneling**: Leverages Ngrok connection bridges to bind local port 8000 and forward traffic through an encrypted live web gateway.
  * **Asynchronous Server Control**: Provisions Uvicorn server structures concurrently inside the background execution loop, preserving interactive notebook resources without resource locking.

In [ ]:
from pyngrok import ngrok
import uvicorn
import asyncio
import sys

# 1. Terminate active ngrok processes safely to prevent port binding conflicts
try:
    ngrok.kill()
except Exception:
    pass

# 2. Authenticate and initialize the public HTTP tunnel configuration securely
NGROK_TOKEN = "3Dm6R5U6lMo9wHHcsojL2abG0J8_7kNjJvwQEkeiX2DDHBLt4"
clean_url = "TUNNEL_CREATION_FAILED"

try:
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(8000)
    clean_url = tunnel.public_url

    print("\n" + "="*60)
    print("SYSTEM IS LIVE! SHARE THIS URL WITH THE MOBILE TEAM:")
    print(f"Base URL: {clean_url}")
    print(f"API Docs (Swagger): {clean_url}/docs")
    print("-" * 60)
    print(f"Predict Endpoint: {clean_url}/predict")
    print(f"Classrooms Search Endpoint: {clean_url}/classrooms/search")
    print(f"Bookmarks Add Endpoint: {clean_url}/bookmarks/add")
    print(f"Bookmarks Fetch Endpoint: {clean_url}/bookmarks/my")
    print("="*60 + "\n")

except Exception as tunnel_err:
    print(f"[CRITICAL] Public tunneling pipeline establishment failed: {str(tunnel_err)}")
    print("[WARNING] API will only be accessible locally via localhost parameters.\n")

# 3. Initialize ASGI server configuration managed dynamically via event loop
try:
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)

    # Inject ASGI serve sequence as a concurrent background task
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())
    print("[STATUS] Asynchronous server loop successfully bound and broadcasting.")
except Exception as server_err:
    print(f"[CRITICAL] Core server infrastructure engine failed to ignite: {str(server_err)}")


SYSTEM IS LIVE! SHARE THIS URL WITH THE MOBILE TEAM:
Base URL: https://lively-fraying-encourage.ngrok-free.dev
API Docs (Swagger): https://lively-fraying-encourage.ngrok-free.dev/docs
------------------------------------------------------------
Predict Endpoint: https://lively-fraying-encourage.ngrok-free.dev/predict
Classrooms Search Endpoint: https://lively-fraying-encourage.ngrok-free.dev/classrooms/search
Bookmarks Add Endpoint: https://lively-fraying-encourage.ngrok-free.dev/bookmarks/add
Bookmarks Fetch Endpoint: https://lively-fraying-encourage.ngrok-free.dev/bookmarks/my

[STATUS] Asynchronous server loop successfully bound and broadcasting.


In [ ]:
main_code = """import io
import os
import sys
import asyncio
import numpy as np
from PIL import Image
import nest_asyncio
import cv2

# FastAPI Toolkits
from fastapi import FastAPI, Header, HTTPException, status, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# Firebase Admin SDK
import firebase_admin
from firebase_admin import credentials, firestore

# ML Frameworks
from ultralytics import YOLO
import easyocr

# ===========================================================================
# 1. CORE APPLICATION & INITIALIZATION TRACKERS
# ===========================================================================
app = FastAPI(
    title="UniWay Centralized Backend API",
    description="Asynchronous cloud core serving computer vision workflows and secure Firestore transaction mappings.",
    version="1.0.0"
)

# Enable Cross-Origin Resource Sharing (CORS) for Mobile Team Connectivity
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global Instance Holders for Secure Defensive Access
db = None
detection_model = None
reader = None

# Custom Pydantic Models for Data Validation
class BookmarkPayload(BaseModel):
    roomId: str

# Helper functions for text normalization and constraints
def normalize_ml_text(text: str) -> str:
    if not text:
        return ""
    text = str(text).strip().lower()
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه").replace("ى", "ي")
    text = text.replace(" ", "").replace("\\t", "").replace("\\n", "")
    return text

def enforce_uqu_room_constraints(digits: str) -> str:
    if not digits:
        return ""
    if len(digits) == 4:
        if digits.endswith('1'):
            return digits[:3]
        elif digits.startswith('1'):
            return digits[1:]
        return digits[:3]
    return digits

def your_custom_preprocessing_pipeline(pil_img):
    try:
        open_cv_image = np.array(pil_img)
        open_cv_image = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2BGR)

        lab = cv2.cvtColor(open_cv_image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, b))
        enhanced = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

        final_img = cv2.fastNlMeansDenoisingColored(enhanced, None, 10, 10, 7, 21)
        return final_img
    except Exception:
        return None

# ===========================================================================
# 2. LIFESPAN RESOURCE INITIALIZATION TRACK (Render & Local Compliant)
# ===========================================================================
@app.on_event("startup")
async def startup_event():
    global db, detection_model, reader
    print("[INIT] Igniting cloud resources initialization sequence...")

    # A. Firebase Initialization Architecture
    try:
        cred_path = os.environ.get("FIREBASE_CREDENTIALS_PATH", "serviceAccountKey.json")
        if not firebase_admin._apps:
            if os.path.exists(cred_path):
                cred = credentials.Certificate(cred_path)
                firebase_admin.initialize_app(cred)
                print("[INIT] Firebase Administrative SDK bound successfully.")
            else:
                print(f"[CRITICAL] Firebase key missing at {cred_path}. Cloud repositories will be offline.")
        db = firestore.client()
    except Exception as fb_err:
        print(f"[CRITICAL] Firebase administrative handshake ruptured: {str(fb_err)}")

    # B. YOLOv8 Model Layer Allocation with Custom Weights
    try:
        model_path = os.environ.get("YOLO_MODEL_PATH", "best_uqu_v1.pt")
        if os.path.exists(model_path):
            detection_model = YOLO(model_path)
            print(f"[INIT] YOLOv8 weight matrix loaded securely from: {model_path}")
        else:
            print(f"[WARNING] Weight matrix file not found at {model_path}. Spatial localization is offline.")
    except Exception as yolo_err:
        print(f"[CRITICAL] Spatial framework configuration locked: {str(yolo_err)}")

    # C. Bilingual OCR Thread Spawning with Enhanced Custom Accuracy Weights
    try:
        reader = easyocr.Reader(['ar', 'en'], gpu=False, model_storage_directory=".", user_network_directory=".")
        print("[INIT] Asynchronous bilingual EasyOCR pipelines generated with dynamic custom weights.")
    except Exception as ocr_err:
        print(f"[CRITICAL] Linguistic pipeline parsing arrays failed to compile: {str(ocr_err)}")

# ===========================================================================
# 3. MACHINE LEARNING INFERENCE ENDPOINT (/predict)
# ===========================================================================
@app.post("/predict")
async def predict_signage(file: UploadFile = File(...)):
    try:
        contents = await file.read()
        try:
            image = Image.open(io.BytesIO(contents)).convert("RGB")
        except Exception as img_err:
            return {
                "status": "error", "error_code": "INVALID_IMAGE_PAYLOAD",
                "message": f"Submitted binary file is corrupted: {str(img_err)}"
            }

        if detection_model is None:
            return {"status": "error", "error_code": "MODEL_UNAVAILABLE", "message": "YOLOv8 weights are unavailable."}

        detection_results = detection_model(image, conf=0.3)
        if len(detection_results[0].boxes) == 0:
            return {"status": "error", "error_code": "NO_SIGNAGE_FOUND", "message": "Signage localization failed."}

        box = detection_results[0].boxes[0].xyxy[0].cpu().numpy().astype(int)
        if (box[2] <= box[0]) or (box[3] <= box[1]):
            return {"status": "error", "error_code": "INVALID_BOUNDING_BOX", "message": "Invalid bounding box localization dimensions."}

        cropped_img = image.crop((box[0], box[1], box[2], box[3]))

        enhanced_numpy = your_custom_preprocessing_pipeline(cropped_img)
        if enhanced_numpy is None:
            enhanced_numpy = np.array(cropped_img)

        raw_text = ""
        if reader is not None:
            ocr_results = reader.readtext(enhanced_numpy, detail=1)
            raw_text = " ".join([res[1] for res in ocr_results]) if ocr_results else ""
            if not raw_text:
                ocr_results_raw = reader.readtext(np.array(cropped_img))
                raw_text = " ".join([res[1] for res in ocr_results_raw]) if ocr_results_raw else ""

        if not raw_text:
            return {"status": "error", "error_code": "OCR_FAILED", "message": "Text extraction failed."}

        hindi_to_eng = {'١':'1','٢':'2','٣':'3','٤':'4','٥':'5','٦':'6','٧':'7','٨':'8','٩':'9','٠':'0'}
        clean_str = str(raw_text).strip()
        for h, e in hindi_to_eng.items():
            clean_str = clean_str.replace(h, e)

        clean_str = clean_str.replace('ا', 'أ').replace('إ', 'أ').replace('آ', 'أ').replace('A', 'أ').replace('a', 'أ')
        clean_str = clean_str.replace('D', 'د').replace('d', 'د')

        raw_digits = "".join(filter(str.isdigit, clean_str))
        letters_part = "".join(filter(lambda x: not x.isdigit(), clean_str)).replace(" ", "")
        digits_part = enforce_uqu_room_constraints(raw_digits)

        if not letters_part:
            letters_part = "أ"

        target_room_id = f"{digits_part}{letters_part}"

        if db is None:
            return {"status": "error", "error_code": "FIRESTORE_OFFLINE", "message": "Database link unavailable."}

        try:
            doc_ref = db.collection('ClassRoom').document(target_room_id)
            doc = doc_ref.get()

            if doc.exists:
                data = doc.to_dict()
                return {
                    "status": "success", "message": "Match Found directly via Document ID Lookup",
                    "data": {
                        "detected_text_raw": raw_text, "processed_room_id": doc.id, "className": doc.id,
                        "buildingId": data.get("buildingId", "Not Specified"), "floorNum": data.get("floorNum", "Not Specified"),
                        "description": data.get("description", "No description available.")
                    }
                }

            normalized_ocr_query = normalize_ml_text(raw_text)
            all_classrooms = db.collection('ClassRoom').stream()

            for room_doc in all_classrooms:
                room_data = room_doc.to_dict()
                db_classname = room_data.get("classname", room_data.get("className", ""))
                normalized_db_classname = normalize_ml_text(db_classname)

                if normalized_db_classname and ((normalized_db_classname in normalized_ocr_query) or (normalized_ocr_query in normalized_db_classname)):
                    return {
                        "status": "success", "message": "Match Found via High-Tolerance Classroom Name Attribute Lookup",
                        "data": {
                            "detected_text_raw": raw_text, "processed_room_id": room_doc.id, "className": db_classname,
                            "buildingId": room_data.get("buildingId", "Not Specified"), "floorNum": room_data.get("floorNum", "Not Specified"),
                            "description": room_data.get("description", "No description available.")
                        }
                    }

            fallback_id = f"{target_room_id} "
            doc_ref_fb = db.collection('ClassRoom').document(fallback_id)
            doc_fb = doc_ref_fb.get()

            if doc_fb.exists:
                data_fb = doc_fb.to_dict()
                return {
                    "status": "success", "message": "Match Found directly via Document ID Lookup (with padding)",
                    "data": {
                        "detected_text_raw": raw_text, "processed_room_id": doc_fb.id, "className": target_room_id,
                        "buildingId": data_fb.get("buildingId", "Not Specified"), "floorNum": data_fb.get("floorNum", "Not Specified"),
                        "description": data_fb.get("description", "No description available.")
                    }
                }

            return {
                "status": "success", "message": "Location could not be identified in database schemas.",
                "data": {
                    "detected_text_raw": raw_text, "processed_room_id": target_room_id, "className": target_room_id,
                    "buildingId": "Unknown", "floorNum": "Unknown", "description": "Location key not found in current database mapping."
                }
            }

        except Exception as db_err:
            return {"status": "error", "error_code": "DATABASE_ERROR", "message": str(db_err)}

    except Exception as e:
        return {"status": "error", "error_code": "SERVER_ERROR", "message": str(e)}

# ===========================================================================
# 4. UNIFIED SEARCH & NAVIGATION ENDPOINT (/classrooms/search)
# ===========================================================================
@app.get("/classrooms/search")
async def search_classrooms(query: str):
    if not query:
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Search context query parameter string cannot be empty.")

    if db is None:
        raise HTTPException(status_code=status.HTTP_503_SERVICE_UNAVAILABLE, detail="Cloud database engine offline.")

    try:
        normalized_user_query = normalize_ml_text(query)
        matching_results = []
        classrooms_stream = db.collection('ClassRoom').stream()

        for doc in classrooms_stream:
            doc_id = str(doc.id)
            room_data = doc.to_dict()

            db_classname = room_data.get("classname", room_data.get("className", ""))
            normalized_id = normalize_ml_text(doc_id)
            normalized_classname = normalize_ml_text(db_classname)

            if (normalized_user_query in normalized_id) or (normalized_user_query in normalized_classname):
                matching_results.append({
                    "roomId": doc_id,
                    "className": db_classname if db_classname else doc_id,
                    "buildingId": room_data.get("buildingId", "Not Specified"),
                    "floorNum": room_data.get("floorNum", "Not Specified"),
                    "description": room_data.get("description", "No description available.")
                })

        return {
            "status": "success",
            "query_evaluated": query,
            "results_count": len(matching_results),
            "data": matching_results
        }
    except Exception as e:
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=f"Search pipeline operational error: {str(e)}")

# ===========================================================================
# 5. USER ISOLATED CLOUD REPOSITORIES (/bookmarks)
# ===========================================================================
@app.get("/bookmarks/my")
async def fetch_my_bookmarks(x_device_id: str = Header(None, alias="x-device-id")):
    if not x_device_id:
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Required hardware identification token 'x-device-id' header is missing.")

    if db is None:
        raise HTTPException(status_code=status.HTTP_503_SERVICE_UNAVAILABLE, detail="Active database connections are currently unavailable.")

    try:
        user_doc_ref = db.collection('Users').document(x_device_id)
        user_doc = user_doc_ref.get()

        if not user_doc.exists:
            return {"status": "success", "device_tracked": x_device_id, "total_bookmarks": 0, "data": []}

        user_data = user_doc.to_dict()
        bookmark_ids = user_data.get('bookmarks', [])

        if not bookmark_ids:
            return {"status": "success", "device_tracked": x_device_id, "total_bookmarks": 0, "data": []}

        detailed_bookmarks = []
        for room_id in bookmark_ids:
            room_ref = db.collection('ClassRoom').document(str(room_id).strip())
            room_doc = room_ref.get()

            if room_doc.exists:
                r_data = room_doc.to_dict()
                detailed_bookmarks.append({
                    "roomId": room_doc.id,
                    "className": r_data.get("classname", r_data.get("className", room_doc.id)),
                    "buildingId": r_data.get("buildingId", "Not Specified"),
                    "floorNum": r_data.get("floorNum", "Not Specified"),
                    "description": r_data.get("description", "No description available.")
                })

        return {
            "status": "success",
            "device_tracked": x_device_id,
            "total_bookmarks": len(detailed_bookmarks),
            "data": detailed_bookmarks
        }
    except Exception as e:
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=f"Failed to pull profile configurations: {str(e)}")

@app.post("/bookmarks/add")
async def add_bookmark(payload: BookmarkPayload, x_device_id: str = Header(None, alias="x-device-id")):
    if not x_device_id:
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Required hardware identification token 'x-device-id' header is missing.")

    if db is None:
        raise HTTPException(status_code=status.HTTP_503_SERVICE_UNAVAILABLE, detail="Database connectivity dropped.")

    try:
        room_ref = db.collection('ClassRoom').document(payload.roomId)
        if not room_ref.get().exists:
            raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail=f"Target room identifier '{payload.roomId}' does not map to any database entity.")

        user_doc_ref = db.collection('Users').document(x_device_id)
        user_doc = user_doc_ref.get()

        if user_doc.exists:
            user_data = user_doc.to_dict()
            current_bookmarks = user_data.get('bookmarks', [])
            if payload.roomId not in current_bookmarks:
                current_bookmarks.append(payload.roomId)
                user_doc_ref.update({'bookmarks': current_bookmarks})
        else:
            user_doc_ref.set({'bookmarks': [payload.roomId]})

        return {
            "status": "success",
            "message": f"Room '{payload.roomId}' securely appended to device profile arrays.",
            "device": x_device_id
        }
    except HTTPException as http_ex:
        raise http_ex
    except Exception as e:
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=f"Failed to complete transactional updates: {str(e)}")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=True)
"""

with open("main.py", "w", encoding="utf-8") as f:
    f.write(main_code)

print("Success! New customized 'main.py' generated with native EasyOCR and YOLOv8 asset alignments.")

Success! New customized 'main.py' generated with native EasyOCR and YOLOv8 asset alignments.


In [ ]:
requirements_code = """fastapi==0.110.0
uvicorn==0.28.0
ultralytics==8.1.0
python-multipart==0.0.9
nest-asyncio==1.6.0
pyngrok==7.1.2
easyocr==1.7.1
firebase-admin==6.5.0
opencv-python-headless==4.9.0.80
numpy==1.26.4
Pillow==10.2.0
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_code)

print("Done! 'requirements.txt' has been successfully generated in your environment root.")

Done! 'requirements.txt' has been successfully generated in your environment root.
